# Import Libraries & Dependencies

In [ ]:
!pip install -q nltk rouge-score pycocoevalcap torch

In [1]:
import json
from pathlib import Path
from tqdm.notebook import tqdm
import torch
import transformers
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import nltk
import pandas as pd
from datasets import Dataset, DatasetDict
import os
import sys
from pathlib import Path
from google.colab import drive
from transformers import EarlyStoppingCallback

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [3]:
drive.mount('/content/gdrive')
# base_dir = "/content/gdrive/MyDrive/Senior/CS 4782/Project/lora/code"
base_dir = "/content/gdrive/MyDrive/Project/lora/code"
sys.path.append(base_dir)
os.chdir(base_dir)

# PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'code' else Path.cwd().resolve()
PROJECT_ROOT = Path(base_dir).resolve().parent
sys.path.append(str(PROJECT_ROOT))

from lora import inject_lora
from generate import generate_batch
import evaluate

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/gdrive/MyDrive/Project/lora/code'

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

gptModel = 'gpt2-medium'

DATASET_FILES = {
    'train': DATA_DIR / f'{gptModel}_trainset.csv',
    'validation': DATA_DIR / f'{gptModel}_devset.csv',
    'test': DATA_DIR / f'{gptModel}_testset_w_refs.csv',
}
LORA_WEIGHTS_FILE = RESULTS_DIR / f'{gptModel}_lora_weights.pt'
PREDICTIONS_FILE = RESULTS_DIR / f'{gptModel}_predictions.txt'
OUTPUT_FILE = RESULTS_DIR / f'{gptModel}_results.json'

# Table 11
BEAM_SIZE = 10
LENGTH_PENALTY = 0.9
NO_REPEAT_NGRAM = 4
MAX_NEW_TOKENS = 128
GEN_BATCH_SIZE = 8


## Load Datasets

In [ ]:
def load_e2e(gptModel):
    base = "https://raw.githubusercontent.com/tuetschek/e2e-dataset/master/"

    def rename(df):
        return df.rename(columns={"mr": "input", "ref": "label"})

    source_files = {
        'train': 'trainset.csv',
        'validation': 'devset.csv',
        'test': 'testset_w_refs.csv',
    }

    for split, filename in source_files.items():
        csv_path = DATASET_FILES[split]
        if not csv_path.exists():
            pd.read_csv(base + filename).to_csv(csv_path, index=False)

    return DatasetDict({
        "train": Dataset.from_pandas(rename(pd.read_csv(DATASET_FILES['train']))),
        "validation": Dataset.from_pandas(rename(pd.read_csv(DATASET_FILES['validation']))),
        "test": Dataset.from_pandas(rename(pd.read_csv(DATASET_FILES['test']))),
    })

dataset = load_e2e(gptModel)
print(f"Dataset files stored in: {DATA_DIR}")

## Inject LoRA

In [ ]:
#Load model and Inject LoRA
model = GPT2LMHeadModel.from_pretrained(gptModel)
model = inject_lora(model, rank=4, alpha=32)
model.eval()
print(f"LoRA injected successfully")

# Count how many trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {trainable:,}")

## Training

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained(gptModel)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

def preprocess(batch):
    input_ids, labels = [], []
    for prompt, target in zip(batch["input"], batch["label"]):
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        target_ids = tokenizer(target + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
        input_ids.append((prompt_ids + target_ids)[:512])
        labels.append(([-100] * len(prompt_ids) + target_ids)[:512])
    return {"input_ids": input_ids, "labels": labels}

train_dataset = dataset["train"].map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names)

val_dataset = dataset["validation"].map(
    preprocess,
    batched=True,
    remove_columns=dataset["validation"].column_names
)

args = transformers.TrainingArguments(
    output_dir=str(RESULTS_DIR),
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=500,
    label_smoothing_factor=0.1,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=1,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = transformers.Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=transformers.DataCollatorForSeq2Seq(
        tokenizer,
        model=model,
        padding=True
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()
torch.save(
    {name: param.detach().cpu() for name, param in model.named_parameters() if param.requires_grad},
    LORA_WEIGHTS_FILE,
)
print(f"Saved LoRA weights to: {LORA_WEIGHTS_FILE}")
model.eval()

In [ ]:
# cell to plot learning curves
import matplotlib.pyplot as plt

# Extract logs from trainer
log_history = trainer.state.log_history

train_steps = []
train_losses = []
eval_steps = []
eval_losses = []

for entry in log_history:
    # Training loss logs usually contain 'loss' but not 'eval_loss'
    if 'loss' in entry and 'eval_loss' not in entry:
        train_steps.append(entry['step'])
        train_losses.append(entry['loss'])
    # Evaluation logs contain 'eval_loss'
    if 'eval_loss' in entry:
        eval_steps.append(entry['step'])
        eval_losses.append(entry['eval_loss'])

plt.figure(figsize=(10, 6))
plt.plot(train_steps, train_losses, label='Training Loss', marker='o')
plt.plot(eval_steps, eval_losses, label='Validation Loss', marker='s')
plt.xlabel('Training Steps')
plt.ylabel('Loss')
plt.title('Learning Curves (LoRA fine‑tuned GPT‑2 medium on E2E)')
plt.legend()
plt.grid(True)
plt.tight_layout()

# Optionally save the figure
fig_path = RESULTS_DIR / 'learning_curves.png'
plt.savefig(fig_path, dpi=150)
print(f'Learning curves saved to: {fig_path}')

plt.show()

# Generate Outputs For Testing

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer.padding_side = 'left'

from collections import OrderedDict

ref_groups = OrderedDict()
for inp, label in zip(dataset['test']['input'], dataset['test']['label']):
    if inp not in ref_groups:
        ref_groups[inp] = []
    ref_groups[inp].append(label)

prompts = list(ref_groups.keys())
# prompts = [f"Input: {inp}\nOutput:" for inp in ref_groups.keys()]
references = list(ref_groups.values())

print(f'Unique MRs (prompts to generate): {len(prompts)}')
print(f'Total reference texts:            {sum(len(r) for r in references)}')
print(f'Average refs per MR:              {sum(len(r) for r in references)/len(prompts):.1f}')
print(f'Sample prompt: {prompts[0][:80]}...')


In [ ]:
# Run generation over all test prompts
all_predictions = []
for i in tqdm(range(0, len(prompts), GEN_BATCH_SIZE), desc='Generating'):
    tokenizer.padding_side = "left"
    raw_batch = prompts[i : i + GEN_BATCH_SIZE]
    batch = tokenizer(
        raw_batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    )
    batch = {k: v.to(device) for k, v in batch.items()}
    all_predictions.extend(generate_batch(model, tokenizer, batch, device))

print(f'\nGenerated {len(all_predictions)} predictions')
print('\nSample outputs:')
for i in range(min(3, len(all_predictions))):
    print(f'  [{i}] Prompt: {prompts[i][:60]}...')
    print(f'       Output: {all_predictions[i]}')
    print()

In [ ]:
# Save predictions
with open(PREDICTIONS_FILE, 'w', encoding='utf-8') as f:
    for pred in all_predictions:
        f.write(pred + '\n')

print(f'Saved predictions to: {PREDICTIONS_FILE}')


# Evaluating Test Results

Computes all five NLG metrics from Table 3 of the paper:

| Metric  | What it measures |
|---------|-----------------|
| BLEU    | N-gram precision (1–4) with brevity penalty |
| NIST    | Like BLEU but weights rarer n-grams more heavily |
| METEOR  | Unigram F-score with stemming + WordNet synonyms (needs `nltk wordnet`) |
| ROUGE-L | Longest common subsequence F-score |
| CIDEr   | TF-IDF-weighted n-gram cosine similarity |

In [ ]:
bleu = evaluate.compute_bleu(all_predictions, references);    print(f'  BLEU:    {bleu}')
nist = evaluate.compute_nist(all_predictions, references);    print(f'  NIST:    {nist}')
meteor = evaluate.compute_meteor(all_predictions, references);  print(f'  METEOR:  {meteor}')
rouge_l = evaluate.compute_rouge_l(all_predictions, references); print(f'  ROUGE-L: {rouge_l}')
cider = evaluate.compute_cider(all_predictions, references);   print(f'  CIDEr:   {cider}')

In [ ]:
results = {
    'num_examples': len(all_predictions),
    'predictions_file': str(PREDICTIONS_FILE),
    'lora_weights_file': str(LORA_WEIGHTS_FILE),
    'BLEU': bleu,
    'NIST': nist,
    'METEOR': meteor,
    'ROUGE-L': rouge_l,
    'CIDEr': cider,
}

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print(f'Results saved to: {OUTPUT_FILE}')